# 32 — BETO con contexto tabular serializado como texto (versión rápida)

**Idea propia.** En vendedores sin historial, el texto es la palanca principal (ablación de la 28). BETO hoy ve solo "título [SEP] garantía". Acá ve además el contexto de la publicación como texto:

`título [SEP] garantía [SEP] categoría: MLA… [SEP] precio: 15 ARS [SEP] tipo: bronze [SEP] compra: buy_it_now [SEP] cantidad: 1 [SEP] vendidas: 0 [SEP] fotos: 5 [SEP] envío gratis: no`

(`notebooks/bert_contexto.py`: misma receta que la Fase Q, `max_length` 128 porque con 64 se truncaba el 57% de las filas y con 128 el 0,6%. Sin `seller_id` ni nada que use el target. `category_id` va como código: el nombre no está en los datos.)

**Por qué puede ayudar.** Los árboles ya tienen precio, tipo, cantidades, fotos y envío, pero ven el texto solo resumido en una probabilidad. BETO puede cruzar **palabras con contexto** ("Lenovo i5" a 45.000 con 1 unidad ≠ "remera" a 45.000). Riesgo: que aprenda lo mismo que los árboles y se equivoque igual que el resto (lección de la Fase U).

**Por qué no es leakage (de la entrada).** Todas las variables están en el test y se calculan fila por fila, sin `condition`.

**⚠️ Sesgo de esta versión rápida (decisión mía, para no esperar 8 h):** solo se entrenaron los **5 modelos globales** (`sin_k`), no los 10 de a pares. Para el fold externo k, la validación recibe `sin_k` (no la vio), pero las filas de train del fold j reciben `sin_j`, **que se entrenó con el fold k**. El HistGB aprende cuánto confiar en BETO con probabilidades de modelos que vieron las etiquetas de la validación: **la CV queda un poco inflada**. En la Fase Q, la versión sesgada dio +0,0012 y la exacta +0,0013 (sesgo chico), pero con un BETO más fuerte podría ser mayor. Por eso:
- se compara contra la variante a **exacta** (0,9435, la referencia pedida) y también contra la variante a **sesgada igual** (BETO común con sus 5 globales), que es la comparación de igual a igual;
- regla de decisión (fijada el 2026-11-04): integrar solo si el promedio final mejora **+0,003 o más y gana 5/5 contra la entrega actual medida igual (variante a sesgada)**; si no, se descarta. Contra la a exacta se muestra como referencia.
- además, en (d): promedio de 3 modelos (CatBoost y HistGB de la entrega + BETO con contexto solo), que **no** pone a BETO con contexto adentro de los árboles.

**Material:** transformer y atención: SIA `S12 - Redes Neuronales - NLP.pdf`; AP `Intro a Redes para datos estructurados.pdf`, slide 25. **No está en el material:** el fine-tuning (ver Fase Q) y serializar variables tabulares como texto.

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 220)
SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1
carpeta_lab = ".cache_laboratorio"


def acc_por_fold(prob, filas=None):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= 0.5).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


sin_historial = np.zeros(n, bool)
for k in range(1, 6):
    va = folds == k
    sin_historial[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()


def globales(carpeta):
    # OOF de los 5 modelos globales: la fila del fold k la predice sin_k (que no la vio)
    p = np.full(n, np.nan)
    for k in range(1, 6):
        d = np.load(os.path.join(carpeta_lab, carpeta, f"sin_{k}.npz"))
        es_train = d["filas"] < n
        p[d["filas"][es_train]] = d["prob"][es_train]
    assert not np.isnan(p).any()
    return p


beto = globales("bert")
beto_ctx = globales("bert_contexto")
logistica = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_logistica.csv"))["prob"].to_numpy()
# Logística por fold externo (filas de train: OOF interna; validación: modelo del train del fold), guardada por la 31
logistica_fold = {k: np.load(os.path.join(carpeta_lab, "textos_31", f"logistica_fold{k}.npy")) for k in range(1, 6)}
for k in range(1, 6):
    assert np.allclose(logistica_fold[k][folds == k], logistica[folds == k])

# Referencia: variante a EXACTA (Fase T)
oof_t = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_combinado.csv"))
assert (oof_t["fold"].to_numpy() == folds).all()
ref_hgb = oof_t["(a) promedio simple"].to_numpy()
ref_final = 0.5 * oof_t["catboost (a) promedio simple"].to_numpy() + 0.5 * ref_hgb
print("Variante a exacta — HistGB:", acc_por_fold(ref_hgb).mean().round(4), "| final:", acc_por_fold(ref_final).mean().round(4),
      "| sin historial (final):", acc_por_fold(ref_final, sin_historial).mean().round(4))

Variante a exacta — HistGB: 0.9425 | final: 0.9435 | sin historial (final): 0.8962


## (a) BETO con contexto solo

OOF de los 5 globales: cada fila la predice el modelo que no la vio, así que **esta comparación no tiene sesgo** (no hay un segundo modelo encima).

In [2]:
solos = {}
for nombre, p in {"logística": logistica, "BETO": beto, "BETO con contexto": beto_ctx,
                  "logística + BETO (columna de a)": (logistica + beto) / 2,
                  "logística + BETO con contexto": (logistica + beto_ctx) / 2,
                  "logística + BETO + BETO con contexto": (logistica + beto + beto_ctx) / 3}.items():
    solos[nombre] = {"total": acc_por_fold(p).mean(), "sin historial": acc_por_fold(p, sin_historial).mean(),
                     "con historial": acc_por_fold(p, ~sin_historial).mean()}
print("BETO con contexto vs BETO:", {k: round(float(v), 4) for k, v in comparar(acc_por_fold(beto_ctx), acc_por_fold(beto)).items()})
print("Correlación BETO vs BETO con contexto:", np.corrcoef(beto, beto_ctx)[0, 1].round(3),
      "| con la logística:", np.corrcoef(logistica, beto_ctx)[0, 1].round(3),
      f"| discrepan con BETO: {((beto >= 0.5) != (beto_ctx >= 0.5)).mean():.1%}")
display(pd.DataFrame(solos).T.round(4))

# Fold por fold: BETO con contexto contra BETO sin contexto (log-loss: Árboles.pdf, slides 12-13; más bajo es mejor)
from sklearn.metrics import log_loss
por_fold_a = pd.DataFrame({
    "acc BETO": acc_por_fold(beto), "acc BETO con contexto": acc_por_fold(beto_ctx),
    "logloss BETO": [log_loss(y_np[folds == k], beto[folds == k]) for k in range(1, 6)],
    "logloss BETO con contexto": [log_loss(y_np[folds == k], beto_ctx[folds == k]) for k in range(1, 6)],
}, index=pd.Index(range(1, 6), name="fold"))
por_fold_a["dif acc"] = por_fold_a["acc BETO con contexto"] - por_fold_a["acc BETO"]
por_fold_a["dif logloss"] = por_fold_a["logloss BETO con contexto"] - por_fold_a["logloss BETO"]
por_fold_a.loc["media"] = por_fold_a.mean()
por_fold_a.round(4)

BETO con contexto vs BETO: {'media': 0.908, 'dif_media': 0.0582, 'dif_desvio': 0.0008, 'folds_gana': 5.0, 'pasa': 1.0}
Correlación BETO vs BETO con contexto: 0.858 | con la logística: 0.816 | discrepan con BETO: 12.0%


,total,sin historial,con historial
logística,0.8414,0.7835,0.8672
BETO,0.8497,0.7926,0.8751
BETO con contexto,0.9080,0.8644,0.9273
logística + BETO (columna de a),0.8589,0.8036,0.8835
logística + BETO con contexto,0.9073,0.8649,0.9261
logística + BETO + BETO con contexto,0.8942,0.8501,0.9137


,acc BETO,acc BETO con contexto,logloss BETO,logloss BETO con contexto,dif acc,dif logloss
fold,,,,,,
1,0.8469,0.9058,0.3525,0.2333,0.0589,-0.1192
2,0.8506,0.9083,0.3454,0.2331,0.0577,-0.1123
3,0.8491,0.9081,0.3501,0.2329,0.0589,-0.1172
4,0.8503,0.9090,0.3432,0.2329,0.0587,-0.1103
5,0.8519,0.9086,0.3428,0.2290,0.0568,-0.1137
media,0.8497,0.9080,0.3468,0.2322,0.0582,-0.1146


## (b) y (c) Columna de texto en el HistGB P7a y en el promedio final

La columna reemplaza a `prob_texto_new` (`passthrough` en la misma posición, `INDICES_CAT` no cambia). Para el fold externo k:
- logística: `logistica_fold[k]` (exacta: OOF interna en el train del fold);
- BETO y BETO con contexto: OOF de los **globales** en todas las filas (**sesgado**, ver arriba).

Candidatas:
- `a sesgada` = (logística + BETO) / 2 con los globales de BETO: referencia de igual a igual;
- `logística + BETO con contexto`;
- `logística + BETO + BETO con contexto`.

In [3]:
def armar(modelo):
    pipeline = armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", "passthrough", ["t_texto"])
    return pipeline


def correr_fold(k, modelo, columna):
    entrena, valida = folds != k, folds == k
    X = train.assign(t_texto=columna)
    params = {"modelo__cat_features": INDICES_CAT} if modelo == "catboost" else {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar(modelo).fit(X[entrena], y[entrena], **params)
    return k, pipeline.predict_proba(X[valida])[:, 1]


def oof(modelo, columna_de_fold):
    prob = np.empty(n)
    for k, p in Parallel(n_jobs=5)(delayed(correr_fold)(k, modelo, columna_de_fold(k)) for k in range(1, 6)):
        prob[folds == k] = p
    return prob


CANDIDATAS = {
    "a sesgada (logística + BETO)": lambda k: (logistica_fold[k] + beto) / 2,
    "logística + BETO con contexto": lambda k: (logistica_fold[k] + beto_ctx) / 2,
    "logística + BETO + BETO con contexto": lambda k: (logistica_fold[k] + beto + beto_ctx) / 3,
}
archivo_oof = os.path.join(carpeta_lab, "oof_beto_contexto.csv")
guardado = pd.read_csv(archivo_oof) if os.path.exists(archivo_oof) else pd.DataFrame(
    {"fold": folds, "y": y_np, "beto_contexto_global": beto_ctx})
for nombre, columna_de_fold in CANDIDATAS.items():
    for modelo in ["histgb", "catboost"]:
        col = f"{modelo} | {nombre}"
        if col not in guardado:
            inicio = time.time()
            guardado[col] = oof(modelo, columna_de_fold)
            guardado.to_csv(archivo_oof, index=False)
            print(f"{col:50s} {(time.time() - inicio) / 60:.1f} min", flush=True)

histgb | a sesgada (logística + BETO)              0.6 min


catboost | a sesgada (logística + BETO)            1.6 min


histgb | logística + BETO con contexto             0.5 min


catboost | logística + BETO con contexto           1.6 min


histgb | logística + BETO + BETO con contexto      0.5 min


catboost | logística + BETO + BETO con contexto    1.5 min


In [4]:
filas = {}
ref_sesg_hgb = guardado["histgb | a sesgada (logística + BETO)"].to_numpy()
ref_sesg_final = 0.5 * guardado["catboost | a sesgada (logística + BETO)"].to_numpy() + 0.5 * ref_sesg_hgb
for nombre in CANDIDATAS:
    p_h = guardado[f"histgb | {nombre}"].to_numpy()
    p_f = 0.5 * guardado[f"catboost | {nombre}"].to_numpy() + 0.5 * p_h
    vs_exacta_h, vs_exacta_f = comparar(acc_por_fold(p_h), acc_por_fold(ref_hgb)), comparar(acc_por_fold(p_f), acc_por_fold(ref_final))
    vs_sesg_f = comparar(acc_por_fold(p_f), acc_por_fold(ref_sesg_final))
    filas[nombre] = {
        "histgb": vs_exacta_h["media"], "histgb_dif_vs_a": vs_exacta_h["dif_media"], "histgb_folds": vs_exacta_h["folds_gana"],
        "histgb_sin_hist": acc_por_fold(p_h, sin_historial).mean(),
        "final": vs_exacta_f["media"], "final_dif_vs_a_exacta": vs_exacta_f["dif_media"], "final_folds_vs_a_exacta": vs_exacta_f["folds_gana"],
        "final_dif_vs_a_sesgada": vs_sesg_f["dif_media"], "final_folds_vs_a_sesgada": vs_sesg_f["folds_gana"],
        "final_sin_hist": acc_por_fold(p_f, sin_historial).mean(),
        "regla (+0,003 y 5/5 vs a sesgada)": vs_sesg_f["dif_media"] >= 0.003 and vs_sesg_f["folds_gana"] == 5,
    }
print("Variante a exacta: HistGB", acc_por_fold(ref_hgb).mean().round(4), "| final", acc_por_fold(ref_final).mean().round(4),
      "| sin historial final", acc_por_fold(ref_final, sin_historial).mean().round(4))
print("Sesgo medido en la referencia (a sesgada − a exacta, promedio final):",
      round(float(acc_por_fold(ref_sesg_final).mean() - acc_por_fold(ref_final).mean()), 5))
tabla = pd.DataFrame(filas).T
tabla.to_csv(os.path.join(carpeta_lab, "beto_contexto.csv"))
display(tabla.round(5))

# Fold por fold del promedio final: cada candidata contra la entrega medida igual (a sesgada) y contra la a exacta
por_fold_b = pd.DataFrame({"entrega (a exacta)": acc_por_fold(ref_final), "entrega (a sesgada)": acc_por_fold(ref_sesg_final)},
                          index=pd.Index(range(1, 6), name="fold"))
for nombre in CANDIDATAS:
    if nombre.startswith("a sesgada"):
        continue
    p_f = 0.5 * guardado[f"catboost | {nombre}"].to_numpy() + 0.5 * guardado[f"histgb | {nombre}"].to_numpy()
    por_fold_b[nombre] = acc_por_fold(p_f)
    por_fold_b[f"dif vs a sesgada | {nombre}"] = por_fold_b[nombre] - por_fold_b["entrega (a sesgada)"]
por_fold_b.loc["media"] = por_fold_b.mean()
por_fold_b.round(5)

Variante a exacta: HistGB 0.9425 | final 0.9435 | sin historial final 0.8962
Sesgo medido en la referencia (a sesgada − a exacta, promedio final): -0.00036


,histgb,histgb_dif_vs_a,histgb_folds,histgb_sin_hist,final,final_dif_vs_a_exacta,final_folds_vs_a_exacta,final_dif_vs_a_sesgada,final_folds_vs_a_sesgada,final_sin_hist,"regla (+0,003 y 5/5 vs a sesgada)"
a sesgada (logística + BETO),0.942414,-0.0001,3,0.894536,0.943171,-0.000357,2,0.0,0,0.895214,False
logística + BETO con contexto,0.943043,0.000529,4,0.895168,0.943386,-0.000143,1,0.000214,3,0.895542,False
logística + BETO + BETO con contexto,0.942814,0.0003,4,0.894248,0.943943,0.000414,4,0.000771,4,0.89666,False


,entrega (a exacta),entrega (a sesgada),logística + BETO con contexto,dif vs a sesgada | logística + BETO con contexto,logística + BETO + BETO con contexto,dif vs a sesgada | logística + BETO + BETO con contexto
fold,,,,,,
1,0.94321,0.94350,0.94414,0.00064,0.94414,0.00064
2,0.94221,0.94264,0.94179,-0.00086,0.94307,0.00043
3,0.94421,0.94336,0.94321,-0.00014,0.94329,-0.00007
4,0.94350,0.94307,0.94343,0.00036,0.94429,0.00121
5,0.94450,0.94329,0.94436,0.00107,0.94493,0.00164
media,0.94353,0.94317,0.94339,0.00021,0.94394,0.00077


## (d) Promedio de 3 modelos: CatBoost + HistGB de la entrega + BETO con contexto solo

Acá BETO con contexto **no entra a los árboles**: se promedian tres probabilidades ya calculadas.
- CatBoost y HistGB: las OOF **exactas** de la variante a (Fase T, `oof_texto_combinado.csv`), o sea, la entrega actual.
- BETO con contexto: OOF de los 5 globales (la fila del fold k la predice `sin_k`).

**Pesos:** grilla de a 0,1 que suman 1. Para evaluar el fold k, se elige la combinación con mejor accuracy en los **otros 4 folds** y se aplica al fold k, rotando (igual que en `16_ensambles` y `23_lightgbm`). Se compara pareado contra la entrega (50/50, pesos fijos de antemano) con el criterio de siempre.

**Por qué esta CV no tiene el sesgo de (b)-(c):**
1. Las tres probabilidades de cada fila son *out-of-fold*: la predijo un modelo que **no la vio** (CatBoost y HistGB con la columna de texto exacta; BETO con contexto con `sin_k`).
2. En (b)-(c) el problema era que un HistGB **se entrena** con probabilidades de BETO de filas de train que salieron de modelos que vieron la validación. Acá no se entrena nada encima de BETO con contexto: solo se eligen **2 números** (los pesos) mirando 4 folds, y el fold que se evalúa no participa de la elección.
3. Matiz (para ser preciso): las OOF de los 4 folds que se usan para elegir los pesos salieron de modelos que sí entrenaron con el fold evaluado. Pasa en cualquier elección de pesos con OOF (16, 23). Con 2 parámetros libres, lo que se puede filtrar por ahí es despreciable; en (b)-(c), el que aprende es un HistGB con cientos de árboles.

In [5]:
p_cat = oof_t["catboost (a) promedio simple"].to_numpy()
p_hgb = ref_hgb
p_ctx = beto_ctx
print("Correlación BETO con contexto vs CatBoost / HistGB:", np.corrcoef(p_ctx, p_cat)[0, 1].round(3), np.corrcoef(p_ctx, p_hgb)[0, 1].round(3))

grilla = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(11) for b in range(11 - a)]
prob_3, pesos_elegidos = np.empty(n), {}
for k in range(1, 6):
    entrena, evalua = folds != k, folds == k
    mejor = max(grilla, key=lambda w: (((w[0] * p_cat + w[1] * p_hgb + w[2] * p_ctx)[entrena] >= 0.5) == y_np[entrena]).mean())
    pesos_elegidos[k] = mejor
    prob_3[evalua] = (mejor[0] * p_cat + mejor[1] * p_hgb + mejor[2] * p_ctx)[evalua]
print("Pesos (CatBoost, HistGB, BETO con contexto) elegidos por fold:", pesos_elegidos)

acc_3, acc_entrega = acc_por_fold(prob_3), acc_por_fold(ref_final)
print("3 modelos vs entrega:", {k: round(float(v), 5) for k, v in comparar(acc_3, acc_entrega).items()})
print("Sin historial: entrega", acc_por_fold(ref_final, sin_historial).mean().round(4),
      "| 3 modelos", acc_por_fold(prob_3, sin_historial).mean().round(4))
guardado["promedio 3 modelos (pesos 4 folds)"] = prob_3
guardado.to_csv(archivo_oof, index=False)
por_fold_d = pd.DataFrame({"entrega": acc_entrega, "3 modelos": acc_3, "dif": acc_3 - acc_entrega,
                           "pesos (cat, hgb, ctx)": [pesos_elegidos[k] for k in range(1, 6)]},
                          index=pd.Index(range(1, 6), name="fold"))
por_fold_d.round(5)

Correlación BETO con contexto vs CatBoost / HistGB: 0.932 0.925
Pesos (CatBoost, HistGB, BETO con contexto) elegidos por fold: {1: (0.6, 0.2, 0.2), 2: (0.3, 0.5, 0.2), 3: (0.3, 0.5, 0.2), 4: (0.5, 0.4, 0.1), 5: (0.6, 0.3, 0.1)}
3 modelos vs entrega: {'media': 0.94333, 'dif_media': -0.0002, 'dif_desvio': 0.00039, 'folds_gana': 2.0, 'pasa': 0.0}
Sin historial: entrega 0.8962 | 3 modelos 0.8963


,entrega,3 modelos,dif,"pesos (cat, hgb, ctx)"
fold,,,,
1,0.94321,0.94307,-0.00014,"(0.6, 0.2, 0.2)"
2,0.94221,0.94236,0.00014,"(0.3, 0.5, 0.2)"
3,0.94421,0.94350,-0.00071,"(0.3, 0.5, 0.2)"
4,0.94350,0.94379,0.00029,"(0.5, 0.4, 0.1)"
5,0.94450,0.94393,-0.00057,"(0.6, 0.3, 0.1)"
